# DX702 Coding Quiz: Week 11

## Imports

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm # For linear regression
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

from scipy.stats import t,skew
from tqdm import tqdm
from scipy.stats import skew
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LinearRegression


In [2]:
plt.rcParams['axes.titlesize']  = 10
plt.rcParams['axes.labelsize']  = 8
plt.rcParams['lines.linewidth'] = 0.5
plt.rcParams['lines.markersize'] = 3
plt.rcParams['axes.edgecolor']  = 'gray'
plt.rcParams['xtick.color']     = 'gray'
plt.rcParams['ytick.color'] = 'gray'
plt.rcParams['xtick.color'] = 'gray'
plt.rcParams['ytick.color'] = 'gray'
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['xtick.labelsize'] = 8

## Functions

In [3]:
def make_error(corr_const, num): 
  
  sigma = 5 * 1 / np.sqrt((1 - corr_const)**2 / (1 - corr_const**2)) 
  err = list() 
  prev = np.random.normal(0, sigma) 
 
  for n in range(num): 
    prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, sigma) 
    err.append(prev) 
  return np.array(err) 

In [4]:
#### FOR QUESTIONS 1 and 2 ####

num = 1000 
# Parameters for the test
# true_event_time = int(num / 2)
# threshold = 1.96 
event_time = int(num / 2) 
 
R_market = np.random.normal(0, 1, num) + np.arange(num) / num 
 
R_target = 2 + R_market + np.random.normal(0, 1, num) + (np.arange(num) == int(num / 2) + 1) * 2 
 
results = sm.OLS(R_target[:event_time], sm.add_constant(R_market[:event_time])).fit() 

# Estimate the coefficients 
alpha, beta = results.params 
 
resid = R_target - results.predict(sm.add_constant(R_market)) 
 
print(resid[event_time + 1] / resid[:event_time].std(ddof = 2))

2.2965894966736315


## Question 1  
**10 Points**  

Which is closest to the probability that the t-test will be able to detect the event at `event_time + 1` with the given code? 

Option A : 0.7

***Option B: 0.5***

Option C : 0.1

Option D : 0.3

In [5]:

# Simulation parameters
num_simulations = 10000
num             = 1000
event_time      = int(num / 2)
threshold       = 1.96  # Two-tailed test at alpha=0.05

# Counter
significant_detections = 0

for _ in range(num_simulations):
    R_market = np.random.normal(0, 1, num) + np.arange(num) / num
    R_target = 2 + R_market + np.random.normal(0, 1, num) + (np.arange(num) == event_time + 1) * 2

    results = sm.OLS(R_target[:event_time], sm.add_constant(R_market[:event_time])).fit()
    resid = R_target - results.predict(sm.add_constant(R_market))
    
    t_stat = resid[event_time + 1] / resid[:event_time].std(ddof=2)
    
    if abs(t_stat) > threshold:
        significant_detections += 1


power = significant_detections / num_simulations
power



0.5163

## Question 2  
**10 Points**  

Use the same code but put `np.random.seed(0)` at the beginning of each loop to ensure that you are <font color='cyan'>performing placebo tests on a fixed dataset</font>. Perform a placebo test by setting the fictitious `event_time` to all possible times, while leaving the event in `R_target` at just the 1 time. The placebo test trains itself on the data leading up to the fictitious event. 

Approximately what fraction of placebo tests seem to detect an event at the fictitious event time? 

Option A : 0.15

Option B: 0.10

Option C : 0.02

***Option D : 0.05***


In [6]:

num             = 1000
true_event_time = int(num / 2)
threshold       = 1.96
false_positives = 0
total_placebos  = 0


R_market = np.random.normal(0, 1, num) + np.arange(num) / num
R_target = 2 + R_market + np.random.normal(0, 1, num) + (np.arange(num) == true_event_time + 1) * 2

# Perform placebo tests
for fictitious_event_time in range(100, num - 2):  # Avoid very early or very late times
    np.random.seed(0)
    # Skip if fictitious event time is the same as true event time
    if fictitious_event_time == true_event_time:
        continue

    # Train model on data before fictitious event
    results = sm.OLS(R_target[:fictitious_event_time], sm.add_constant(R_market[:fictitious_event_time])).fit()
    alpha, beta = results.params

    # Compute residuals
    resid = R_target - results.predict(sm.add_constant(R_market))

    # Compute t-statistic at fictitious event time
    t_stat = resid[fictitious_event_time + 1] / resid[:fictitious_event_time].std(ddof=2)

    # Check if t-statistic exceeds threshold
    if abs(t_stat) > threshold:
        false_positives += 1

    total_placebos += 1

# Compute fraction of false positives
false_positive_rate = false_positives / total_placebos
print(f"Fraction of placebo tests that falsely detect an event: {false_positive_rate:.4f}")



Fraction of placebo tests that falsely detect an event: 0.0580


## Question 3  
**10 Points**  

Do the same placebo test, but this time only run the test 20 times before and 20 times after the actual event. On average (over many runs of the code), <font color='cyan'>what fraction of the 40 placebo tests get a higher t-value than the actual event?</font> This time, adjust `np.random.seed()` to represent a different dataset when needed. 


With enough data, the coefficients of X and Y are closest to: 

Option A: 0.05

Option B: 0.35

Option C: 0.25

***Option D: 0.15***

In [7]:
import numpy as np
import statsmodels.api as sm

# Parameters
num = 1000
true_event_time = int(num / 2)
num_simulations = 1000
placebo_range = 20  # 20 before and 20 after
t_value_comparisons = []

# Run multiple simulations
for seed in range(num_simulations):
    np.random.seed(seed)
    
    # Generate data
    R_market = np.random.normal(0, 1, num) + np.arange(num) / num
    R_target = 2 + R_market + np.random.normal(0, 1, num) + (np.arange(num) == true_event_time + 1) * 2

    # Estimate model before the true event
    results = sm.OLS(R_target[:true_event_time], sm.add_constant(R_market[:true_event_time])).fit()
    alpha, beta = results.params
    resid = R_target - results.predict(sm.add_constant(R_market))
    true_t_value = resid[true_event_time + 1] / resid[:true_event_time].std(ddof=2)

    # Placebo tests
    placebo_t_values = []
    for offset in range(-placebo_range, placebo_range + 1):
        if offset == 0:
            continue  # skip the true event
        fictitious_event_time = true_event_time + offset
        if fictitious_event_time <= 1 or fictitious_event_time >= num - 1:
            continue  # skip invalid indices

        # Train on data before fictitious event
        results_placebo = sm.OLS(R_target[:fictitious_event_time], sm.add_constant(R_market[:fictitious_event_time])).fit()
        resid_placebo = R_target - results_placebo.predict(sm.add_constant(R_market))
        t_val = resid_placebo[fictitious_event_time + 1] / resid_placebo[:fictitious_event_time].std(ddof=2)
        placebo_t_values.append(t_val)

    # Compare placebo t-values to true t-value
    fraction_higher = np.mean(np.abs(placebo_t_values) > np.abs(true_t_value))
    t_value_comparisons.append(fraction_higher)

# Average fraction of placebo tests with higher t-value
average_fraction = np.mean(t_value_comparisons)

# Estimate coefficients with large data
np.random.seed(42)
large_num = 100000
R_market_large = np.random.normal(0, 1, large_num) + np.arange(large_num) / large_num
R_target_large = 2 + R_market_large + np.random.normal(0, 1, large_num)
results_large = sm.OLS(R_target_large, sm.add_constant(R_market_large)).fit()
alpha_large, beta_large = results_large.params

print(f"Average fraction of placebo tests with higher t-value than true event: {average_fraction:.4f}")
print(f"Estimated coefficients with large data: alpha = {alpha_large:.4f}, beta = {beta_large:.4f}")

average_fraction, alpha_large, beta_large



Average fraction of placebo tests with higher t-value than true event: 0.1461
Estimated coefficients with large data: alpha = 1.9984, beta = 1.0052


(0.1461, 1.9983823604813276, 1.0051873679814403)

## Question 4  
**10 Points**  

Do the same thing as in question 2, but this time use `make_error` with `corr_const = 0.9` to generate the error for `R_target` instead of `np.random.normal`. Approximately what fraction of placebo tests seem to detect an event at the fictitious event time? 

Consider before attempting this: Would you expect this kind of dataset, where errors are not independent, to result in more or fewer false positives in the placebo tests? 

Option A
0.45

Option B
0.25

Option C
0.65

***Option D
0.05***


<font color='plum'> YES. More false positives are to be expected in placebo tests when the dataset has autocorrelated errors (equivalently, when the errors are not independent.)

#### Standard errors are underestimated
- Autocorrelation causes the residuals to be more predictable and less variable than they appear.
- This leads to smaller estimated standard errors, which in turn inflates the t-statistics.


#### Inflated t-statistics increase false positives
- With inflated t-values, more placebo tests will appear statistically significant even when there's no real event.
- This leads to a higher Type I error rate (false positives).

In [8]:
np.random.seed(0) # Fix the overall dataset generation

num = 1000
true_event_index = int(num / 2) + 1 # The actual event is at index 501
corr_const = 0.9

R_market_fixed_q4 = np.random.normal(0, 1, num) + np.arange(num) / num
# Generate autocorrelated error for R_target
autocorr_error = make_error(corr_const, num)
R_target_fixed_q4 = 2 + R_market_fixed_q4 + autocorr_error + (np.arange(num) == true_event_index) * 2

detections_placebo_q4 = 0
total_placebo_tests_q4 = 0
alpha_level = 0.05

for fictitious_event_time in range(2, num - 1):
    # Train on data up to fictitious_event_time
    results_placebo_q4 = sm.OLS(R_target_fixed_q4[:fictitious_event_time], sm.add_constant(R_market_fixed_q4[:fictitious_event_time])).fit()
    # Corrected typo: sm_add_constant -> sm.add_constant
    resid_placebo_q4 = R_target_fixed_q4 - results_placebo_q4.predict(sm.add_constant(R_market_fixed_q4))

    # Calculate t-statistic at the fictitious event time + 1
    if fictitious_event_time + 1 < num:
        t_statistic_placebo_q4 = resid_placebo_q4[fictitious_event_time + 1] / resid_placebo_q4[:fictitious_event_time].std(ddof = 2)

        df_placebo_q4 = fictitious_event_time - 2
        if df_placebo_q4 > 0:
            critical_t_value_placebo_q4 = t.ppf(1 - alpha_level/2, df=df_placebo_q4)
            # Corrected typo: t_statistic_placeblo_q4 -> t_statistic_placebo_q4
            if np.abs(t_statistic_placebo_q4) > critical_t_value_placebo_q4:
                detections_placebo_q4 += 1
            total_placebo_tests_q4 += 1

fraction_detected_q4 = detections_placebo_q4 / total_placebo_tests_q4 if total_placebo_tests_q4 > 0 else 0
print(f"Fraction of placebo tests detecting an event with autocorrelated errors: {fraction_detected_q4:.3f}")


/opt/anaconda3/lib/python3.12/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/anaconda3/lib/python3.12/site-packages/numpy/core/_methods.py:198: RuntimeWarning: divide by zero encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Fraction of placebo tests detecting an event with autocorrelated errors: 0.042


In [10]:
import numpy as np
import statsmodels.api as sm

# Function to generate autocorrelated errors
def make_error(corr_const, num):
    sigma = 5 * 1 / np.sqrt((1 - corr_const)**2 / (1 - corr_const**2))
    err = []
    prev = np.random.normal(0, sigma)
    for _ in range(num):
        prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, sigma)
        err.append(prev)
    return np.array(err)

# Parameters
num             = 1000
true_event_time = int(num / 2)
threshold       = 1.96

# Function to perform placebo tests
def placebo_test(use_autocorrelated_error=False):
    false_positives = 0
    for event_time in range(100, num - 100):
        np.random.seed(0)
        R_market = np.random.normal(0, 1, num) + np.arange(num) / num
        if use_autocorrelated_error:
            error = make_error(0.9, num)
        else:
            error = np.random.normal(0, 1, num)
        R_target = 2 + R_market + error + (np.arange(num) == true_event_time + 1) * 2

        results = sm.OLS(R_target[:event_time], sm.add_constant(R_market[:event_time])).fit()
        resid = R_target - results.predict(sm.add_constant(R_market))
        test_stat = resid[event_time + 1] / resid[:event_time].std(ddof=2)

        if abs(test_stat) > threshold:
            false_positives += 1

    return false_positives / (num - 200)

# Run placebo tests
fraction_independent    = placebo_test(use_autocorrelated_error=False)
fraction_autocorrelated = placebo_test(use_autocorrelated_error=True)

print(f"Fraction of false positives with independent errors: {fraction_independent:.3f}")
print(f"Fraction of false positives with autocorrelated errors: {fraction_autocorrelated:.3f}")



Fraction of false positives with independent errors: 0.044
Fraction of false positives with autocorrelated errors: 0.040
